# XGBoost Model Selection

This notebook evaluates an XGBoost classifier for the binary diabetes target using 5-fold stratified cross-validation. Metrics are computed directly from each validation fold.

In [1]:
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import recall_score, precision_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

In [2]:
# ==========================================
# 1. Load Training Data ONLY
# ==========================================
train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns=['Diabetes_01'])
y_train = train_df['Diabetes_01']

In [3]:
# ==========================================
# 2. Setup Stratified Cross-Validation
# ==========================================
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
# ==========================================
# 3. Build XGBoost Model
# ==========================================
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [5]:
# ==========================================
# 4. Run 5-Fold Cross-Validation Across Thresholds
# ==========================================

thresholds = [0.5, 0.4, 0.3, 0.25, 0.2]

fold_metrics = []
fold_predictions = []

for fold_number, (train_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    estimator = clone(xgb_model)

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_valid = y_train.iloc[valid_idx]

    estimator.fit(X_fold_train, y_fold_train)

    y_valid_proba = estimator.predict_proba(X_valid)[:, 1]

    for threshold in thresholds:
        y_valid_pred = (y_valid_proba >= threshold).astype(int)

        fold_metrics.append({
            "Fold": fold_number,
            "Threshold": threshold,
            "Validation Recall": recall_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation Precision": precision_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F1": f1_score(y_valid, y_valid_pred, pos_label=1, zero_division=0),
            "Validation F2": fbeta_score(y_valid, y_valid_pred, beta=2, pos_label=1, zero_division=0),
            "Validation AUPRC": average_precision_score(y_valid, y_valid_proba),
            "Validation AUROC": roc_auc_score(y_valid, y_valid_proba),
            "Predicted Positive Rate": y_valid_pred.mean(),
        })

    fold_predictions.append(pd.DataFrame({
        "Fold": fold_number,
        "y_valid": y_valid.to_numpy(),
        "y_valid_proba": y_valid_proba,
    }))

xgb_fold_metrics_df = pd.DataFrame(fold_metrics)
xgb_cv_predictions_df = pd.concat(fold_predictions, ignore_index=True)

xgb_fold_metrics_df.round(6)

,Fold,Threshold,Validation Recall,Validation Precision,Validation F1,Validation F2,Validation AUPRC,Validation AUROC,Predicted Positive Rate
0,1,0.50,0.782547,0.317600,0.451825,0.605318,0.429004,0.815314,0.376309
1,1,0.40,0.865183,0.277331,0.420024,0.607599,0.429004,0.815314,0.476459
2,1,0.30,0.926091,0.246026,0.388771,0.596385,0.429004,0.815314,0.574895
3,1,0.25,0.951380,0.231727,0.372680,0.586865,0.429004,0.815314,0.627037
4,1,0.20,0.966162,0.216463,0.353685,0.570788,0.429004,0.815314,0.681681
5,2,0.50,0.784328,0.315699,0.450192,0.604779,0.434292,0.816731,0.379437
6,2,0.40,0.871950,0.278593,0.422269,0.611480,0.434292,0.816731,0.478009
7,2,0.30,0.924488,0.247049,0.389905,0.597051,0.434292,0.816731,0.571522
8,2,0.25,0.946215,0.232201,0.372894,0.585894,0.434292,0.816731,0.622358
9,2,0.20,0.962422,0.217833,0.355258,0.571634,0.434292,0.816731,0.674772


In [6]:
# ==========================================
# 5. Display Selected Cross-Validation Metrics
# ==========================================

xgb_cv_summary_df = (
    xgb_fold_metrics_df
    .groupby("Threshold", as_index=False)
    .agg({
        "Validation Recall": "mean",
        "Validation Precision": "mean",
        "Validation F1": "mean",
        "Validation F2": "mean",
        "Validation AUPRC": "mean",
        "Validation AUROC": "mean",
        "Predicted Positive Rate": "mean",
    })
    .rename(columns={
        "Validation Recall": "Validation Recall Mean",
        "Validation Precision": "Validation Precision Mean",
        "Validation F1": "Validation F1 Mean",
        "Validation F2": "Validation F2 Mean",
        "Validation AUPRC": "Validation AUPRC Mean",
        "Validation AUROC": "Validation AUROC Mean",
        "Predicted Positive Rate": "Predicted Positive Rate Mean",
    })
)

default_threshold = 0.5
default_scores = xgb_cv_summary_df.loc[
    xgb_cv_summary_df["Threshold"] == default_threshold
].copy()
default_scores.insert(0, "Selection Rule", "Default threshold")

best_f2_scores = xgb_cv_summary_df.loc[
    [xgb_cv_summary_df["Validation F2 Mean"].idxmax()]
].copy()
best_f2_scores.insert(0, "Selection Rule", "Max F2 threshold")

y_valid = xgb_cv_predictions_df["y_valid"]
y_valid_proba = xgb_cv_predictions_df["y_valid_proba"]
fpr, tpr, roc_thresholds = roc_curve(y_valid, y_valid_proba)
best_tpr_fpr_index = (tpr - fpr).argmax()
best_tpr_fpr_threshold = roc_thresholds[best_tpr_fpr_index]
best_tpr_fpr_pred = (y_valid_proba >= best_tpr_fpr_threshold).astype(int)

best_tpr_fpr_scores = pd.DataFrame([{
    "Selection Rule": "Max TPR-FPR threshold",
    "Threshold": best_tpr_fpr_threshold,
    "Validation Recall Mean": recall_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation Precision Mean": precision_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F1 Mean": f1_score(y_valid, best_tpr_fpr_pred, pos_label=1, zero_division=0),
    "Validation F2 Mean": fbeta_score(y_valid, best_tpr_fpr_pred, beta=2, pos_label=1, zero_division=0),
    "Validation AUPRC Mean": average_precision_score(y_valid, y_valid_proba),
    "Validation AUROC Mean": roc_auc_score(y_valid, y_valid_proba),
    "Predicted Positive Rate Mean": best_tpr_fpr_pred.mean(),
    "TPR - FPR": tpr[best_tpr_fpr_index] - fpr[best_tpr_fpr_index],
}])

xgb_selected_metrics_df = pd.concat(
    [default_scores, best_f2_scores, best_tpr_fpr_scores],
    ignore_index=True
)

xgb_selected_metrics_df.round(6)


,Selection Rule,Threshold,Validation Recall Mean,Validation Precision Mean,Validation F1 Mean,Validation F2 Mean,Validation AUPRC Mean,Validation AUROC Mean,Predicted Positive Rate Mean,TPR - FPR
0,Default threshold,0.500000,0.781387,0.315768,0.449774,0.603426,0.433184,0.814954,0.377965,NaN
1,Max F2 threshold,0.400000,0.865513,0.277831,0.420635,0.608207,0.433184,0.814954,0.475819,NaN
2,Max TPR-FPR threshold,0.505509,0.776507,0.318319,0.451537,0.602935,0.432862,0.814934,0.372590,0.476733
